<a href="https://colab.research.google.com/github/dhinesh-66/flyrank-ml-internship/blob/main/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

**Lane:** Content Refresh Prioritization

**ML Task Type:** Ranking

**Why?**

My goal is to rank webpages based on their priority for content refresh. Instead of simply deciding whether a page should be refreshed or not, the model will order pages from highest to lowest priority. This helps the content team focus on updating the pages that are most likely to improve performance.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

The dataset does not contain a direct column called **refresh_priority**, so I will use a proxy target.

I will use the **trend_direction** and **trend_pct** columns together with metrics such as **impressions**, **clicks**, **sessions**, **CTR**, **average position**, and **content age** to estimate which webpages should be refreshed first.

This proxy is based on observed performance data rather than a manually created rule.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

Since this is a ranking problem, the model should place the highest-priority webpages near the top of the recommendation list.

A suitable ranking metric is **NDCG (Normalized Discounted Cumulative Gain)** because it measures how well the recommended order matches the ideal order.

The output should help the content team decide which webpages to refresh first.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [11]:
import os
os.listdir("/content")

['.config', 'sample_data']

In [12]:
!git clone https://github.com/dhinesh-66/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 114 (delta 29), reused 80 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.85 MiB | 9.15 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [13]:
import os
os.listdir("/content")

['.config', 'flyrank-ml-internship', 'sample_data']

In [14]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv


In [15]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

In [16]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [17]:
df.shape

(30000, 44)

In [18]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [20]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [21]:
df[['content_id',
    'content_type',
    'search_volume',
    'clicks_90d',
    'sessions_90d',
    'trend_direction',
    'trend_pct']].head()

,content_id,content_type,search_volume,clicks_90d,sessions_90d,trend_direction,trend_pct
0,content_304f48230142,keyword article,10.0,29,17,down,-41.4
1,content_a1fb4e703a9e,keyword article,90.0,7,9,down,-57.7
2,content_9aa793d4d895,keyword article,0.0,11,11,down,-60.9
3,content_331d6c4de07b,keyword article,10.0,58,78,stable,-13.8
4,content_d99b7a2d90ca,keyword article,0.0,24,145,down,-34.7


## 4. The unit of analysis

The unit of analysis is **one webpage (content item)**.

Each row in the dataframe represents one piece of content and includes information such as search volume, clicks, sessions, engagement, content age, and trend information.

The model will evaluate each webpage independently and assign a refresh priority ranking.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule such as "refresh every page older than one year" ignores many important factors.

A webpage may be old but still perform well, while a newer page may be losing traffic and require immediate attention.

Machine learning can combine multiple signals such as search volume, clicks, sessions, CTR, engagement rate, average position, content age, and trend direction to identify patterns that are difficult to capture with a single rule.

This provides better decision support for the content team.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.